In [7]:
import sys
from core import LADTransferTreeBoost, LSTransferTreeBoost, MTransferTreeBoost
import pandas as pd
from sklearn.model_selection import train_test_split
import xgboost as xgb
import numpy as np
from utils import * #only needed for xgboost
import itertools
import matplotlib.pyplot as plt
from baselines import *

In [8]:
test_size_list = [0.8] #0.8or 0.93
target_columns = ['Volume']
seed_list = [1]

In [9]:
predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

#random_size = np.random.randint(57, 59)
#predictor_columns = np.random.choice(predictor_columns, size = random_size)
predictor_columns

['pzabovezmean',
 'pzabove2',
 'zq5',
 'zq10',
 'zq15',
 'zq20',
 'zq25',
 'zq30',
 'zq35',
 'zq40',
 'zq45',
 'zq50',
 'zq55',
 'zq60',
 'zq65',
 'zq70',
 'zq75',
 'zq80',
 'zq85',
 'zq90',
 'zq95',
 'zpcum1',
 'zpcum2',
 'zpcum3',
 'zpcum4',
 'zpcum5',
 'zpcum6',
 'zpcum7',
 'zpcum8',
 'zpcum9']

In [ ]:

v_list = [0.1]
source_tree_size_list = [1]
target_tree_size_list = [1]
k_list = [0.01]
m_0_list = [0.7]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    v_list,
    source_tree_size_list,
    target_tree_size_list,
    k_list,
    m_0_list
))

# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:
    for test_size in test_size_list:
        for target_column in target_columns:

            #data from Svedala
            data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])


            #evaluate and rain on latvia instead (keep naming for simplicity)
            #data from latvia target
            data_latvia = pd.read_csv(r'datasets/rs_lettland.csv', index_col=[0])
            train_size = int((1-test_size)*len(data_latvia))
            data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
            data_train, data_temp = train_test_split(data_latvia, test_size=test_size, random_state=seed)
            data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

            #"General" base dataset (to use for transfer)
            X_source_train = np.array(data_sweden[predictor_columns])
            y_source_train = np.array(data_sweden[target_column])

            #Specific train and test set
            X_target_train = np.array(data_train[predictor_columns])
            y_target_train = np.array(data_train[target_column])

            X_target_val = np.array(data_val[predictor_columns])
            y_target_val = np.array(data_val[target_column])

            X_target_test = np.array(data_test[predictor_columns])
            y_target_test = np.array(data_test[target_column])

            print(len(X_target_train), len(X_target_val), len(X_target_test))
            for config in param_grid:
                v, source_tree_size, target_tree_size, k, m_0 = config


                #Test for all methods!!!!

                method = f'LSTransferTreeBoost'
                fiter = LADTransferTreeBoost(epochs=1000, v=v, source_tree_size=source_tree_size, 
                                            target_tree_size=target_tree_size, k=k, m_0=m_0)
                fiter.fit(X_target_train, y_target_train, X_source_train, y_source_train, val_x=X_target_val, val_y=y_target_val, early_stopping_rounds=8, show_curves = True)
                rmse = fiter.evaluate(X_target_test, y_target_test, metric = 'rmse')
                val_rmse = fiter.evaluate(X_target_val, y_target_val, metric = 'rmse')
                mae = fiter.evaluate(X_target_test, y_target_test, metric = 'mae')
                val_mae = fiter.evaluate(X_target_val, y_target_val, metric = 'mae')
                 
    

C:\Users\Dag Bjornberg\AppData\Local\Temp\ipykernel_26616\3447211327.py:25: DtypeWarning: Columns (4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_sweden = pd.read_csv(r'datasets/rs_sweden.csv', index_col=[0])


380 760 760


In [ ]:
rmse

np.float64(83.60158493939925)

In [ ]:
a = np.array([-10,1])
np.sign(a)

array([-1,  1])